In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *

In [ ]:
import numpy as np
import pandas as pd

# --- Black-Box Graph Generator ---

# Randomize total nodes (close to 10,000, but not fixed)
n = np.random.randint(9500, 10501)

# Randomly choose the community balance (fraction for community 0 between 0.2 and 0.8)
balance = np.random.uniform(0.3, 0.7)
n_comm0 = int(np.round(balance * n))
n_comm1 = n - n_comm0

# Create opaque community assignments
Z = np.zeros(n, dtype=int)
indices = np.arange(n)
np.random.shuffle(indices)
Z[indices[:n_comm0]] = 0
Z[indices[n_comm0:]] = 1

# Internally generate hidden "attributes" for each node that drive edge formation.
# (The specifics remain undisclosed, but they are engineered to produce communities with similar behaviors.)
hidden_factors = np.random.randn(n, 3)  # extra dimensions to hide clarity

# Apply a non-trivial transformation on the hidden factors that creates subtle differences.
transformed = np.tanh(hidden_factors + np.random.uniform(-0.5, 0.5, size=hidden_factors.shape))

# --- Create edges with a black-box probabilistic model ---
edges = []
num_candidates = round(n / 100)  # each node considers a small, random subset of all other nodes

for i in range(n):
    # Randomly choose candidate neighbors (excluding itself)
    candidates = np.random.choice(np.delete(np.arange(n), i), size=num_candidates, replace=False)
    for j in candidates:
        # Compute a hidden measure that determines edge likelihood.
        metric = np.abs(np.sum(transformed[i] - transformed[j])) / 3
        # Internally adjust probabilities, making intra- and inter-community behavior hard to separate.
        if Z[i] == Z[j]:
            prob = np.exp(-metric) * np.random.uniform(0.8, 1.2)
        else:
            prob = np.exp(-metric) * np.random.uniform(0.8, 1.2)
        # A slight bias to allow a moderate number of edges overall.
        if np.random.rand() < prob * 0.4:
            weight = np.random.rand()
            # Ensure a consistent ordering of nodes to avoid duplicate edges.
            edge = tuple(sorted((i, j)))
            edges.append((edge[0], edge[1], weight))

# Remove potential duplicate edges and prepare the DataFrame
unique_edges = list(set(edges))
df_edges = pd.DataFrame(unique_edges, columns=["node1", "node2", "weight"])

# Save results to CSV files
#df_edges.to_csv("synthetic_graph_edges.csv", index=False)
#df_communities = pd.DataFrame({"node": np.arange(n), "community": Z})
#df_communities.to_csv("synthetic_graph_communities.csv", index=False)

print("Black-box synthetic graph generated:")
print(f" - Total nodes: {n}")
print(f" - Community 0: {n_comm0}, Community 1: {n_comm1}")
print(f" - Total edges: {len(unique_edges)}")
print("Output files: 'synthetic_graph_edges.csv' and 'synthetic_graph_communities.csv'")


Black-box synthetic graph generated:
 - Total nodes: 10175
 - Community 0: 6190, Community 1: 3985
 - Total edges: 284799
Output files: 'synthetic_graph_edges.csv' and 'synthetic_graph_communities.csv'


In [6]:
for emb_mode, p22 in product(EMB_MODES[:1], P22S):
	print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
	metrics = {}
	for rho, pi in product(RHOS, PIS):
		metrics[(rho, pi)] = {}
		for model, model_params in MODELS_AND_PARAMS:
			m = model(rho, pi, model_params, p22 = p22)
			A, Z = m(42)
			metrics[(rho, pi)][m] = {}
			for t in TRANSFORMS:
			#for t in TRANSFORMS_THR_QTL:
				print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
				metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

	plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
	for rho, pi in product(RHOS, PIS):
		subfolder = f"Tranforms_Beta_Lognormal"
		#subfolder = f"Threshold_Quantile_Beta_Lognormal"
		plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Identity
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Opposite
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Logarithmic
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Rank
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Identity
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Opposite
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Logarithmic
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Rank
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Identity
Simulating

In [ ]:
for emb_mode, p22 in product(EMB_MODES[:1], P22S):
	print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
	metrics = {}
	for rho, pi in product(RHOS, PIS):
		metrics[(rho, pi)] = {}
		for model, model_params in product([lognormWSBM], [(1, 1), (0.5, 1), (1, 0.5), (0.1, 0.5)]):
		#for model, model_params in product([lognormWSBM], [(0.05, 0.25), (0.05, 0.9), (0.5, 0.05), (0.2, 0.25)]):
			m = model(rho, pi, model_params, p22 = p22)
			A, Z = m(42)
			metrics[(rho, pi)][m] = {}
			#for t in TRANSFORMS:
			for t in TRANSFORMS_THR_QTL:
				print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
				metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

	plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
	for rho, pi in product(RHOS, PIS):
		#subfolder = f"Tranforms_Lognormal"
		#subfolder = f"Threshold_Quantile_Lognormal"
		#subfolder = f"Threshold_Quantile_Lognormal_Peculiarities"
		plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ = 0.25, σ₂₂ = 1.0
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = 0.05, σ₁₂ 

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.9
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.9
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold 

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25


...

Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.9
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.9
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.05, σ₁₂ = 0.9
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
L

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.5, σ₁₂ = 0.05
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.5, σ₁₂ = 0.05
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.5, σ₁₂ = 0.05
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25


c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.25)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.5)
Lognorm-WSBM: μ = -2.33
σ₁₁ = σ₂₂ = 0.2, σ₁₂ = 0.25


In [2]:
emb_mode = 'sqrt-scaled'
p22 = 'fixed'
n_batch = 2

path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}/Grids"
metrics_g = {}
metrics_g_1st_layer = {}
for rho, pi, model in RHOS_PIS_MODELS:
	file = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	grids_stacked = [np.load(f"{file}/{b}.npz") for b in range(n_batch)]
	metrics_g[(rho, pi, model)] = {}
	metrics_g_1st_layer[(rho, pi, model)] = {}
	for t in TRANSFORMS:
		metrics_g[(rho, pi, model)][t] = {}
		metrics_g[(rho, pi, model)][t]['std'] = {}
		metrics_g_1st_layer[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			g_stack = np.concatenate([g[f'{t.id}_{metric}'] for g in grids_stacked], axis = -1)
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std
			metrics_g_1st_layer[(rho, pi, model)][t][metric] = g_stack[:, :, 0]

metrics_g = aggregate_metrics(metrics_g)
metrics_g_1st_layer = aggregate_metrics(metrics_g_1st_layer)

metrics_g = best_transform_metrics(metrics_g)

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Computation\ExtraMetrics.py:75: RuntimeWarning: divide by zero encountered in log
  return np.log(pred / (true + eps))
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Computation\ExtraMetrics.py:75: RuntimeWarning: divide by zero encountered in log
  return np.log(pred / (true + eps))
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_meth

In [4]:
plotter.plot_scatter_Rand_vs_Chernoff(metrics_g_1st_layer, n_points_ratio_displayed=0.2)

In [5]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

In [6]:
# Prendre moins de place première ligne

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_bias_heatmap(rho, pi, model, t, m, log = True)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-pa

In [3]:
# Rand moyen, Regret moyen pour Best transform overall
# Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area C-estim-Best Transform) over 8 graphs (TreeMap)
# Pour chaque Transform élue Best Transform
#	  - Rand moyen + Regret moyen
#     Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area Rand-Best Transform) over 8 graphs (TreeMap)
#     Rand moyen pour chaque Best transform

# Average(Rand) over 8 graphs for 4 transforms + Best

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plotter.plot_best_transform_heatmaps(rho, pi, model, m)

In [ ]:
import plotly.express as px

# Extraction des métriques
m = metrics_g['C_graph-Best Transform']
# Valeurs globales
regret_area       = m['Regret Area']
rand_avg_total    = m['Rand Avg']
regret_avg_total  = m['Regret Avg']

# Labels
root_label = (
    f"C_graph-Best Transform<br>"
    f"Rand Avg : {rand_avg_total:.3f}<br>"
    f"Regret Avg : {regret_avg_total:.3f}"
)
child_labels = [
    (
        f"Where Regret > 0<br>"
        f"Rand Avg : {m['Rand Avg on Positive Regret']:.3f}<br>"
        f"Regret Avg : {m['Regret Avg on Positive Regret']:.3f}"
    ),
    (
        f"Where Regret == 0<br>"
        f"Rand Avg : {m['Rand Avg on Null Regret']:.3f}<br>"
        f"Regret Avg : 0.000"
    )
]

# Construction des listes
names   = [root_label] + child_labels
parents = [""] + [root_label, root_label]
values  = [1.0, regret_area, 1 - regret_area]

fig = px.treemap(
    names       = names,
    parents     = parents,
    values      = values,
    branchvalues= "total"
)

# Centrage et customisation du texte
fig.update_traces(
    # %{label} = votre texte défini plus haut ; %{value} = la surface du rectangle
    texttemplate  = "%{label}<br>Area : %{value:.3f}",
    textposition  = "middle center",
    hovertemplate = None  # si vous voulez supprimer le tooltip par défaut
)

fig.show()

In [7]:
emb_mode = 'sqrt-scaled'
p22 = 'fixed'
fixed_param = '11'

plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

In [ ]:
path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}/Lines"

metrics_l = {}
for p in linspace_exclusive(0, 1, 4):
	metrics_l[p] = {}
	for rho, pi, model in RHOS_PIS_MODELS:
		param_str = f'{model.param_name}{fixed_param}'
		file = f"{model.__name__}_{rho}_{pi}_{param_str}_{p}".replace('.', '')
		lines_stacked = np.load(f"{path}/{file}.npz")
		metrics_l[p][(rho, pi, model)] = {}
		metrics_l[p][(rho, pi, model)]['fixed_param'] = lines_stacked['fixed_param']
		for tid, t in TRANSFORMS_MAP.items():
			metrics_l[p][(rho, pi, model)][t] = {}
			metrics_l[p][(rho, pi, model)][t]['std'] = {}
			for metric in METRICS_ID:
				#line = local_weighted_average(lines_stacked[f'{tid}_{metric}'])
				mean = np.mean(lines_stacked[f'{tid}_{metric}'], axis = -1)
				std = np.std(lines_stacked[f'{tid}_{metric}'], axis = -1)
				metrics_l[p][(rho, pi, model)][t][metric] = mean
				metrics_l[p][(rho, pi, model)][t]['std'][metric] = std
			
for p in linspace_exclusive(0, 1, 4):
	for rho, pi, model in RHOS_PIS_MODELS:
		m = metrics_l[p][(rho, pi, model)]
		metrics_l[p][(rho, pi, model)] = best_transform_metrics(m)

In [9]:
for p in linspace_exclusive(0, 1, 4):
	for rho, pi, model in RHOS_PIS_MODELS:
		m_l = metrics_l[p][(rho, pi, model)]
		plotter.plot_best_transform_lines_light(rho, pi, model, m_l)